# Phase 2 — PyTorch Star Center Detection (`StarCenterUNetMini`)

Thin orchestration notebook calling reusable modules in `src/constellation_detection/`.
Compatible with **Google Colab GPU** (`requirements-colab.txt`) and **Local CPU** (`venv/`).

In [ ]:
# 0. Optional Colab Bootstrap (no hardcoded Drive mounts or personal paths)
# Run these lines in a fresh Google Colab GPU runtime if needed:
# !git clone https://github.com/MyTNinja/constellation-detection.git
# %cd constellation-detection
# %pip install -r requirements-colab.txt
# !python scripts/prepare_phase2_pilot_dataset.py --num-fields 24 --cutout-px 1024
import os

if os.environ.get('COLAB_RELEASE_TAG'):
    print('Detected Google Colab runtime.')
    print('If starting from a fresh runtime, clone the repo, run %pip install -r requirements-colab.txt,')
    print('and prepare data/ via scripts/prepare_phase2_pilot_dataset.py before running the cells below.')

In [ ]:
# 1-3. Environment inspection, dependency verification & runtime device probe
import json
from torch.utils.data import DataLoader
from constellation_detection.artifacts import RunArtifactManager, build_run_id
from constellation_detection.config import ProjectPaths
from constellation_detection.datasets import StarHeatmapDataset, audit_pilot_dataset, load_annotation_record, load_split_manifest
from constellation_detection.detection.dl import build_model, detect_stars_dl
from constellation_detection.evaluation.matching import compute_average_precision, evaluate_phase1_baselines_on_manifest, match_detections_to_ground_truth, select_best_threshold
from constellation_detection.image.io import load_image
from constellation_detection.runtime import collect_env_snapshot, make_dataloader_worker_init, resolve_device, seed_everything
from constellation_detection.training import StarCenterLoss, TrainingConfig, run_training

env_snap = collect_env_snapshot()
print(json.dumps(env_snap, indent=2))
device, dev_info = resolve_device(preferred='auto', require_gpu=False)
print('Resolved device:', dev_info)

In [ ]:
# 4-8. Dataset loading, validation, annotation inspection, visualization & split check
import matplotlib.pyplot as plt
import numpy as np

paths = ProjectPaths.from_paths()
manifest_path = paths.splits_dir / 'dataset_manifest_v1.0.json'
manifest = load_split_manifest(manifest_path)
audit_report = audit_pilot_dataset(manifest_path, paths.data_dir)
print('Dataset audit verdict:', audit_report['verdict'], '| split_counts:', audit_report['split_counts'])
assert audit_report['verdict'] != 'FAIL', audit_report['issues']

cfg = TrainingConfig()
seed_everything(cfg.seed, deterministic=cfg.deterministic)
train_ds = StarHeatmapDataset(manifest_path, paths.data_dir, split='train', crop_size=cfg.crop_size, augment=True, sigma_px=cfg.sigma_px, seed=cfg.seed)
val_ds = StarHeatmapDataset(manifest_path, paths.data_dir, split='val', crop_size=cfg.crop_size, augment=False, sigma_px=cfg.sigma_px, seed=cfg.seed)
print('Train crops:', len(train_ds), '| Val crops:', len(val_ds))

sample_item = train_ds[0]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(sample_item['image'][0].numpy(), cmap='gray')
axes[0].set_title(f'Train crop: {sample_item["image_id"]}')
axes[1].imshow(sample_item['heatmap'][0].numpy(), cmap='magma')
axes[1].set_title('Rendered Gaussian Star Target')
plt.tight_layout()
plt.show()

In [ ]:
# 9-14. Model construction, loss check, optimizer/scheduler, training, validation & checkpoints
RUN_TRAINING = False  # Explicitly set True only on approved GPU training runs

model = build_model(cfg.model_name, {'in_channels': cfg.in_channels, 'base_channels': cfg.base_channels})
print('Parameters:', sum(p.numel() for p in model.parameters()))
criterion = StarCenterLoss(alpha=cfg.focal_alpha, beta=cfg.focal_beta, lambda_offset=cfg.lambda_offset)
worker_fn = make_dataloader_worker_init(cfg.seed)
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, worker_init_fn=worker_fn)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)

run_id = build_run_id('p2', 'stardet', 'unetmini', cfg.dataset_version, cfg.seed)
artifact_mgr = RunArtifactManager(paths.runs_dir, run_id)
artifact_mgr.write_json_atomic('run_metadata.json', {'run_id': run_id, 'env': env_snap, 'device': dev_info.to_dict()})

if RUN_TRAINING:
    summary = run_training(model, train_loader, val_loader, cfg, artifact_mgr, dev_info)
    print('Training completed:', summary['best_epoch'], summary['best_val_loss'])
else:
    print('Dry-run verified cleanly. Set RUN_TRAINING = True to start GPU training.')

In [ ]:
# 15-17. Frozen Phase 1 baseline benchmark, validation-selected threshold & transfer bundle
baseline_rows = evaluate_phase1_baselines_on_manifest(manifest_path, paths.data_dir, split='test', tolerance_px=3.0)
for row in baseline_rows:
    print(row)

if RUN_TRAINING:
    import math
    from constellation_detection.artifacts import load_weights_artifact

    best_weights_path = summary.get('best_weights_path')
    if not best_weights_path:
        raise RuntimeError('No best weights artifact recorded in training summary.')

    ckpt = load_weights_artifact(best_weights_path)
    eval_model = build_model(ckpt['model_name'], ckpt.get('model_kwargs', {}))
    eval_model.load_state_dict(ckpt['state_dict'], strict=True)
    eval_model.to(device)
    eval_model.eval()

    val_samples = [s for s in manifest['samples'] if str(s['split']) == 'val']
    val_preds = []
    val_gt = []
    for sample in val_samples:
        img = load_image(paths.data_dir / sample['image_relpath'])
        ann = load_annotation_record(paths.data_dir / sample['annotation_relpath'])
        det = detect_stars_dl(
            img,
            eval_model,
            device=device,
            threshold=0.05,
            min_dist_px=0.0,
            max_candidates=500,
            in_channels=cfg.in_channels,
            detector_name=ckpt['model_name'],
        )
        val_preds.append(det['stars_xyf'])
        val_gt.append(ann['stars'])

    best_thresh, val_summary = select_best_threshold(val_preds, val_gt, tolerance_px=3.0)
    print('Validation-selected threshold:', best_thresh, val_summary)

    test_samples = [s for s in manifest['samples'] if str(s['split']) == 'test']
    test_preds_raw = []
    test_gt = []
    test_latencies = []
    for sample in test_samples:
        img = load_image(paths.data_dir / sample['image_relpath'])
        ann = load_annotation_record(paths.data_dir / sample['annotation_relpath'])
        det = detect_stars_dl(
            img,
            eval_model,
            device=device,
            threshold=0.05,
            min_dist_px=0.0,
            max_candidates=500,
            in_channels=cfg.in_channels,
            detector_name=ckpt['model_name'],
        )
        test_preds_raw.append(det['stars_xyf'])
        test_gt.append(ann['stars'])
        test_latencies.append(float(det['latency_ms']))

    pr_auc = compute_average_precision(test_preds_raw, test_gt, tolerance_px=3.0)
    tp_tot = fp_tot = fn_tot = sat_tot = sat_tp = 0
    all_errs = []
    for pred, gt in zip(test_preds_raw, test_gt):
        filtered = pred[pred[:, 2] >= best_thresh] if pred.size > 0 else pred
        m = match_detections_to_ground_truth(filtered, gt, tolerance_px=3.0)
        tp_tot += m['tp']
        fp_tot += m['fp']
        fn_tot += m['fn']
        sat_tot += m['saturated_total']
        sat_tp += m['saturated_tp']
        all_errs.extend(m['matched_errors_px'])

    n_test = len(test_samples)
    prec = float(tp_tot / (tp_tot + fp_tot)) if (tp_tot + fp_tot) > 0 else 0.0
    rec = float(tp_tot / (tp_tot + fn_tot)) if (tp_tot + fn_tot) > 0 else 0.0
    f1 = float(2.0 * prec * rec / (prec + rec)) if (prec + rec) > 0.0 else 0.0
    rmse = math.sqrt(sum(e * e for e in all_errs) / len(all_errs)) if all_errs else None
    dl_summary = {
        'detector': ckpt['model_name'],
        'split': 'test',
        'num_images': n_test,
        'tolerance_px': 3.0,
        'selected_threshold': best_thresh,
        'precision': round(prec, 4),
        'recall': round(rec, 4),
        'f1': round(f1, 4),
        'pr_auc': round(pr_auc, 4),
        'fppi': round(fp_tot / float(n_test), 2) if n_test > 0 else 0.0,
        'rmse_px': round(rmse, 4) if rmse is not None else None,
        'saturated_recall': round(sat_tp / float(sat_tot), 4) if sat_tot > 0 else None,
        'mean_latency_ms': round(sum(test_latencies) / float(n_test), 2) if n_test > 0 else 0.0,
        'device': str(device),
    }
    print('PyTorch test summary:', dl_summary)

    eval_report = {
        'phase1_baselines': baseline_rows,
        'val_threshold_selection': val_summary,
        'pytorch_test_metrics': dl_summary,
    }
    artifact_mgr.write_json_atomic('metrics/evaluation_summary.json', eval_report)
    bundle_path = artifact_mgr.export_run_bundle(include_trainstate=False)
    print('Exported inference bundle:', bundle_path)
else:
    print('Skipping PyTorch evaluation & bundle export because RUN_TRAINING=False.')